# **End to End Generation Model using KV Cache**

## What's actually wasteful - The Theory

At iteration 5, the sequence is `[i, love, code, and, i, love]` — 6 tokens. The model computes:

**For every layer:**
- Embedding for all 6 tokens
- Q, K, V projections for all 6 tokens
- Attention scores (a 6×6 matrix of token-pair scores)
- Attention output for all 6 positions
- FFN for all 6 positions
- LayerNorm for all 6 positions

**Of which we use:** only position 5's final output.

Now look at what changes between iteration 4 and iteration 5:

**Iteration 4** processed `[i, love, code, and, i]` — 5 tokens. It already computed embeddings, Q, K, V, attention, FFN, LayerNorm for positions 0 through 4.

**Iteration 5** processes `[i, love, code, and, i, love]` — 6 tokens. The token "love" at position 5 is genuinely new. But for positions 0–4, we're **recomputing identical work** — same input tokens, same weights, same outputs.

So at iteration 5, the wasted work is:
- Embeddings for positions 0–4 (computed 5 times across iterations)
- Q, K, V for positions 0–4 (computed 5 times each)
- Attention output for positions 0–4 (which we don't even use)
- FFN and LayerNorm for positions 0–4 (also unused)

Only position 5's computation is genuinely new and needed.

By iteration N of generation, we've recomputed token 0's full pipeline N times. That's the quadratic cost we mentioned earlier — O(N²) total work to generate N tokens.

---

## Now the question — what can we actually cache?

Look carefully at what each token needs at iteration 5.

The new token "love" (position 5) needs to compute attention using the attention formula:

```
attention_output = softmax(Q @ K.T / sqrt(d_k)) @ V
```

Position 5's attention output depends on:
- **Its own Q** (Q at position 5)
- **K and V of every previous position** (positions 0–4)

So the new token needs:
- A new Q for itself
- Access to K and V of all previous tokens

**Crucial observation:** K and V at positions 0–4 were already computed in iteration 4. They depend only on tokens 0–4, which haven't changed. **The same K and V will be produced again.**

So instead of recomputing them, we **cache** them. That's literally the entire idea behind the "KV cache" — store K and V from previous iterations, reuse them.

---

## What we cache and what we don't

This is where the name matters. We cache K and V — not Q. Why not Q?

Think about what Q is for. Q at position 5 asks: *"As the token at position 5, what information am I looking for from other tokens?"* That's a question only the new token needs to ask.

What about Q at positions 0–4 from previous iterations? Those Qs asked questions like *"As the token at position 0, what am I looking for?"* — and those questions were used during their iterations to compute attention for *those* positions. But at iteration 5, we don't need attention output at position 0, 1, 2, 3, or 4. We only need position 5's attention output.

**So old Qs are useless. We never compute them again. We don't cache them either.**

The cache stores:
- K from every previous token (the new token needs to attend TO them)
- V from every previous token (the new token needs to retrieve their values)

That's it. Q is computed fresh, only for the new token, each iteration.

---

## The new computation per iteration (with cache)

Once the cache is in place, generating one new token requires:

1. Compute embedding for **only the new token** (1 vector, not the whole sequence)
2. Compute Q, K, V for **only the new token** (one Q, one K, one V per layer)
3. Append the new K and V to the cache
4. Compute attention: new Q against ALL cached K and V → one attention output
5. Run FFN on **only the new token's** output (not the whole sequence)
6. Pass through final norm and output head → next-token prediction

Each step is O(1) per layer (proportional to one token's work), regardless of how long the sequence already is. The only "growing" cost is the attention dot product, which is O(seq_len) — but that's optimal because the new token really does need to look at all previous tokens.

**Total cost to generate N tokens drops from O(N²) to O(N).** Huge.

---

In [ ]:
import torch
import torch.nn as nn
import math


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model   = d_model
        self.num_heads = num_heads
        self.d_k       = d_model // num_heads

        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, kv_cache=None):
        """
        x:        input vectors. Shape (seq_len, d_model).
                  - During training/prefill: seq_len = full sequence length
                  - During cached generation: seq_len = 1 (just the new token)
        kv_cache: dict with keys 'K' and 'V' holding cached tensors,
                  or None for the first call (no cache yet).
        Returns:
            attention output, updated kv_cache
        """
        seq_len = x.shape[0]

        # Compute Q, K, V for whatever tokens we received (could be 1, could be many)
        Q = self.W_Q(x)
        K = self.W_K(x)
        V = self.W_V(x)

        # If there's a cache, prepend the cached K and V
        if kv_cache is not None:
            K = torch.cat([kv_cache['K'], K], dim=0)
            V = torch.cat([kv_cache['V'], V], dim=0)

        # Update the cache (this gets passed back out)
        new_kv_cache = {'K': K, 'V': V}

        # Now K and V represent the FULL sequence so far
        # But Q is only for the NEW tokens we just received
        full_len = K.shape[0]

        # Split into heads — Q has seq_len rows, K and V have full_len rows
        Q = Q.view(seq_len,  self.num_heads, self.d_k).permute(1, 0, 2)
        K = K.view(full_len, self.num_heads, self.d_k).permute(1, 0, 2)
        V = V.view(full_len, self.num_heads, self.d_k).permute(1, 0, 2)

        # Attention: Q (heads, seq_len, d_k) @ K.T (heads, d_k, full_len) → (heads, seq_len, full_len)
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)
        attention_weights = torch.softmax(scores, dim=-1)
        head_outputs = attention_weights @ V  # (heads, seq_len, d_k)

        # Merge heads back
        head_outputs = head_outputs.permute(1, 0, 2).contiguous().view(seq_len, self.d_model)

        return self.W_O(head_outputs), new_kv_cache

### Explanation

The signature changed: `forward` now accepts an optional `kv_cache` and returns an updated one alongside the output.

**`if kv_cache is not None:`** — When the cache exists, we concatenate the cached K and V with the newly computed K and V. The new tokens' K and V get appended at the bottom (because `dim=0` is the sequence dimension).

So after this concatenation:
- `Q` has shape `(seq_len, d_model)` — only for the new tokens
- `K` and `V` have shape `(full_len, d_model)` — for ALL tokens, cached + new

This asymmetry is the whole point. We only need Q for the new tokens (they're the ones asking "what should I attend to?"), but we need K and V from everything (so the new tokens can attend to all previous tokens).

**`new_kv_cache = {'K': K, 'V': V}`** — store the full K, V (cached + new) so the next iteration has even more cached. The cache grows by `seq_len` rows per iteration.

**The shape difference matters in the head split.** Look carefully:

```python
Q = Q.view(seq_len, ...)     # Q has seq_len rows
K = K.view(full_len, ...)    # K has full_len rows
V = V.view(full_len, ...)    # V has full_len rows
```

In the un-cached version, all three had the same length. Now they differ. This is fine because matrix multiplication doesn't require them to match — `Q @ K.T` produces a `(seq_len, full_len)` score matrix where row i column j is "how much does new-token i attend to position j of the full sequence?"

**`scores` shape**: `(num_heads, seq_len, full_len)`. Each new token gets one row of scores against all positions in the full sequence.

**Why no causal masking?** Notice we didn't add a mask preventing tokens from attending to "future" tokens. That's because during generation, there ARE no future tokens — we're appending one at a time. The new token's attention naturally only sees what's already in the cache. This is one elegant side benefit of cached generation: the masking concern from training disappears.

---

The new attention module is a superset of the old one — it does the same thing when no cache is provided, and additionally supports caching when one is.
This means:

Training:
- pass kv_cache=None → identical to before
- First generation iteration (prefill): pass kv_cache=None → process the full prompt, get an initial cache to return
- Subsequent generation iterations (decode): pass the previous cache + only the new token → cached behavior

One module handles all three modes cleanly. The naming convention in production LLMs is:

- **Prefill**: the first forward pass that processes the prompt and builds the initial KV cache
- **Decode**: each subsequent forward pass that adds one new token and uses the cache

In [ ]:
# Existing Feed Forward Network:
# No change in FFN block
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.layer1 = nn.Linear(d_model, d_ff)
        self.relu   = nn.ReLU()
        self.layer2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.layer2(self.relu(self.layer1(x)))

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        self.attention    = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, kv_cache=None):
        """
        x:        input vectors (seq_len, d_model)
        kv_cache: this block's cache (dict with 'K' and 'V'), or None
        Returns:
            output vectors, updated kv_cache for this block
        """
        # Attention sub-block
        attn_out, new_kv_cache = self.attention(self.norm1(x), kv_cache=kv_cache)
        x = x + attn_out

        # FFN sub-block (unchanged — FFN has no cache)
        ff_out = self.feed_forward(self.norm2(x))
        x = x + ff_out

        return x, new_kv_cache

**1. The `forward` signature now accepts and returns a cache.**

```python
def forward(self, x, kv_cache=None):
    ...
    return x, new_kv_cache
```

The block doesn't *do* anything with the cache itself — it just relays it to its `attention` submodule and passes back what comes out.

**2. The attention call now passes the cache through:**

```python
attn_out, new_kv_cache = self.attention(self.norm1(x), kv_cache=kv_cache)
```

Notice the norm still happens before attention — we normalize the input first, then feed it (along with the cache) into attention. The cache flow is independent of the norm/residual flow.

**3. The FFN doesn't get cached.**

Look carefully — only the attention sub-block touches the cache. The FFN sub-block runs as before:

```python
ff_out = self.feed_forward(self.norm2(x))
x = x + ff_out
```

---

**Why does FFN not need a cache?** Here's a really important conceptual point. The FFN is a **per-token operation**. Each token's FFN output depends only on that single token's vector — there's no cross-token mixing in the FFN. (Cross-token mixing happens only in attention.)

So when we feed only one new token into the block during decode, the FFN simply computes that one token's output. There's nothing to "cache" because we're never recomputing FFN for old tokens — once a token leaves the block, its FFN output was already used to update the residual stream in its own iteration.

In contrast, attention is **inter-token** — the new token's attention output depends on all previous tokens' K and V. That's exactly why K and V need to be cached.

So the rule is simple: **anything that mixes tokens together needs caching. Per-token operations don't.**

In [ ]:
# Positional Encoding:
def positional_encoding(seq_len, d_model):
    pe = torch.zeros(seq_len, d_model)
    for pos in range(seq_len):
        for i in range(0, d_model, 2):
            denom = 10000 ** (i / d_model)
            pe[pos, i]     = math.sin(pos / denom)
            pe[pos, i + 1] = math.cos(pos / denom)
    return pe

In [ ]:
class MiniModel(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers):
        super().__init__()
        self.embedding   = nn.Embedding(vocab_size, d_model)
        self.blocks      = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_ff)
            for _ in range(num_layers)
        ])
        self.final_norm  = nn.LayerNorm(d_model)
        self.output_head = nn.Linear(d_model, vocab_size)

    def forward(self, token_ids, kv_caches=None, start_pos=0):
        """
        token_ids: tokens to process THIS call. Shape (seq_len,).
                   - During training: full sequence
                   - During generation prefill: full prompt
                   - During generation decode: just the 1 new token
        kv_caches: list of per-block caches, or None.
                   Length = num_layers, one cache dict per block.
        start_pos: the position INDEX of the first token in token_ids
                   within the full sequence. Needed for positional encoding.
        Returns:
            logits, updated kv_caches
        """
        seq_len = token_ids.shape[0]

        # If no caches were passed, initialize as a list of Nones (one per block)
        if kv_caches is None:
            kv_caches = [None] * len(self.blocks)

        # Step 1: Embed
        x = self.embedding(token_ids)

        # Step 2: Add positional encoding
        # Build PE for the positions these tokens occupy in the FULL sequence
        full_pe = positional_encoding(start_pos + seq_len, x.shape[1])
        pe_for_these_tokens = full_pe[start_pos : start_pos + seq_len]
        x = x + pe_for_these_tokens

        # Step 3: Pass through each block, threading caches per-block
        new_kv_caches = []
        for block, block_cache in zip(self.blocks, kv_caches):
            x, new_block_cache = block(x, kv_cache=block_cache)
            new_kv_caches.append(new_block_cache)

        # Step 4: Final norm and output head
        x = self.final_norm(x)
        logits = self.output_head(x)

        return logits, new_kv_caches

### Explanation

Three things changed at the model level: the signature accepts/returns per-block caches, the positional encoding accounts for the token's actual position in the full sequence, and the forward loop threads each block's own cache.

---

**Change 1: One cache per block.**

```python
if kv_caches is None:
    kv_caches = [None] * len(self.blocks)
```

Each transformer block has its **own** K and V values to cache. Block 1 sees the original embedding → produces its K, V. Block 2 sees Block 1's *output* → produces *different* K, V (because the input changed). Block 3 sees Block 2's output → again different K, V.

So the cache structure looks like this for a 3-block model:

```
kv_caches = [
    {'K': K_for_block_1, 'V': V_for_block_1},     ← cache for block 1's attention
    {'K': K_for_block_2, 'V': V_for_block_2},     ← cache for block 2's attention
    {'K': K_for_block_3, 'V': V_for_block_3},     ← cache for block 3's attention
]
```

If you tried to share a single cache across all blocks, you'd be feeding block 2 with block 1's K, V — which is wrong. Block 2 needs its own K, V derived from its own input (which is block 1's output).

**Memory implication of this:** in a 96-layer model like GPT-3, the KV cache stores 96 separate (K, V) pairs per token. This is why KV cache memory becomes a major concern in production — it grows linearly with sequence length AND linearly with depth. For long sequences in deep models, the cache can be larger than the model weights themselves.

---

**Change 2: Position-aware positional encoding.**

```python
full_pe = positional_encoding(start_pos + seq_len, x.shape[1])
pe_for_these_tokens = full_pe[start_pos : start_pos + seq_len]
x = x + pe_for_these_tokens
```

Positional encoding tells the model **where** each token sits in the sequence. During training and prefill, this is straightforward — we have a sequence starting at position 0, so we just generate PE for positions 0 through seq_len-1.

But during decode, we're feeding only the new token. Say we've already generated 5 tokens and are about to feed the 6th. That 6th token sits at **position 5** in the full sequence — not position 0.

If we just did `positional_encoding(1, d_model)`, we'd compute PE for position 0 and add it to the 6th token. The model would think the 6th token is at the start of the sequence, completely confusing its positional signal.

The fix: track `start_pos`, which says "where in the full sequence does my current input begin?" Then slice the positional encoding accordingly.

**Example walkthrough:**

```
Iteration 1 (prefill): token_ids = [I, love],  start_pos = 0
   full_pe is generated for positions 0–1
   slice [0:2] is used  → PE for positions 0, 1 ✓

Iteration 2 (decode): token_ids = [code],  start_pos = 2
   full_pe is generated for positions 0–2
   slice [2:3] is used  → PE for position 2 only ✓

Iteration 3 (decode): token_ids = [and],  start_pos = 3
   full_pe is generated for positions 0–3
   slice [3:4] is used  → PE for position 3 only ✓
```

Each new token gets the correct positional encoding for its actual location in the sequence.

---

**Change 3: Thread per-block caches through the forward loop.**

```python
new_kv_caches = []
for block, block_cache in zip(self.blocks, kv_caches):
    x, new_block_cache = block(x, kv_cache=block_cache)
    new_kv_caches.append(new_block_cache)
```

`zip(self.blocks, kv_caches)` pairs up each block with its corresponding cache slot. Block 1 gets `kv_caches[0]`, block 2 gets `kv_caches[1]`, etc.

Inside the loop, `x` flows through the blocks as before (each block's output feeds the next). But each block independently updates its own cache, and we collect those updated caches into `new_kv_caches`.

At the end, we return both `logits` (for picking the next token) and `new_kv_caches` (for the next iteration to reuse).

---

In [ ]:
def _pick_next_token(logits_1d, temperature, do_sample):
    """Helper: convert a 1D logits vector into a single token ID."""
    scaled = logits_1d / temperature
    probs  = torch.softmax(scaled, dim=-1)
    if do_sample:
        return torch.multinomial(probs, num_samples=1).item()
    else:
        return probs.argmax().item()

In [ ]:
def generate_with_cache(
    model,
    prompt_text,
    word_to_id,
    id_to_word,
    max_new_tokens=10,
    temperature=1.0,
    do_sample=False,
):
    """
    Generate text using KV cache for efficiency.
    """
    model.eval()

    # Step 1: Tokenize the prompt
    prompt_tokens = prompt_text.split()
    token_ids = torch.tensor([word_to_id[t] for t in prompt_tokens])

    print(f"Starting prompt: {prompt_text!r} → tokens: {token_ids.tolist()}")

    generated_ids = token_ids.tolist()

    with torch.no_grad():

        # ─── Prefill phase ──────────────────────────────────────────
        # Process the entire prompt at once; this builds the initial cache.
        logits, kv_caches = model(
            token_ids,
            kv_caches=None,
            start_pos=0,
        )

        # Pick the first generated token (from the LAST position of the prompt)
        next_token_id = _pick_next_token(logits[-1], temperature, do_sample)
        generated_ids.append(next_token_id)
        print(f"  Prefill done. First generated token: '{id_to_word[next_token_id]}'")

        # ─── Decode phase ───────────────────────────────────────────
        # From here on, we feed ONE token at a time, using and growing the cache.
        for step in range(max_new_tokens - 1):
            current_pos = len(generated_ids) - 1   # position the new token occupies

            input_tensor = torch.tensor([next_token_id])

            logits, kv_caches = model(
                input_tensor,
                kv_caches=kv_caches,
                start_pos=current_pos,
            )

            # logits has shape (1, vocab_size) — just one row, for the one token we fed
            next_token_id = _pick_next_token(logits[-1], temperature, do_sample)
            generated_ids.append(next_token_id)
            print(f"  Step {step+2}: '{id_to_word[next_token_id]}'")

    return " ".join(id_to_word[i] for i in generated_ids)

In [ ]:
# ── Training (same as before, just with num_layers) ────────────────────

text   = "the cat sat on the mat because it was tired"
tokens = text.lower().split()
vocab  = sorted(set(tokens))
word_to_id = {w: i for i, w in enumerate(vocab)}
id_to_word = {i: w for w, i in word_to_id.items()}
token_ids = torch.tensor([word_to_id[t] for t in tokens])

print("Number of tokens:", len(tokens))
print("Vocabulary size:", len(vocab))
print("\nVocabulary:")
print(vocab)
print("\nToken IDs:")
print(token_ids)

inputs  = token_ids[:-1]
targets = token_ids[1:]

# Config
vocab_size = len(vocab)
d_model    = 8
num_heads  = 2
d_ff       = 32
num_layers = 3            # ← NEW! 3 stacked transformer blocks

model     = MiniModel(vocab_size, d_model, num_heads, d_ff, num_layers)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn   = nn.CrossEntropyLoss()

print(f"Model has {sum(p.numel() for p in model.parameters())} parameters")
print(f"Stack depth: {num_layers} transformer blocks\n")

for step in range(250):
    logits, _ = model(inputs) # Unpack logits and ignore kv_caches
    loss   = loss_fn(logits, targets) # Remove extra torch.tensor([logits])

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 20 == 0:
        print(f"Step {step:3d} | Loss: {loss.item():.4f}")

# Final predictions
print("\n=== Final Predictions ===")
with torch.no_grad():
    logits, _ = model(inputs) # Unpack logits and ignore kv_caches
    predicted_ids = logits.argmax(dim=-1)

for i, (inp, pred, target) in enumerate(zip(inputs, predicted_ids, targets)):
    input_word  = id_to_word[inp.item()]
    pred_word   = id_to_word[pred.item()]
    target_word = id_to_word[target.item()]
    correct     = "✓" if pred.item() == target.item() else "✗"
    print(f"  [{correct}] Input: '{input_word}' → Predicted: '{pred_word}' (should be: '{target_word}')")

Number of tokens: 10
Vocabulary size: 9

Vocabulary:
['because', 'cat', 'it', 'mat', 'on', 'sat', 'the', 'tired', 'was']

Token IDs:
tensor([6, 1, 5, 4, 6, 3, 0, 2, 8, 7])
Model has 2689 parameters
Stack depth: 3 transformer blocks

Step   0 | Loss: 2.2482
Step  20 | Loss: 0.5233
Step  40 | Loss: 0.1167
Step  60 | Loss: 0.0373
Step  80 | Loss: 0.0195
Step 100 | Loss: 0.0129
Step 120 | Loss: 0.0094
Step 140 | Loss: 0.0073
Step 160 | Loss: 0.0058
Step 180 | Loss: 0.0048
Step 200 | Loss: 0.0040
Step 220 | Loss: 0.0034
Step 240 | Loss: 0.0030

=== Final Predictions ===
  [✓] Input: 'the' → Predicted: 'cat' (should be: 'cat')
  [✓] Input: 'cat' → Predicted: 'sat' (should be: 'sat')
  [✓] Input: 'sat' → Predicted: 'on' (should be: 'on')
  [✓] Input: 'on' → Predicted: 'the' (should be: 'the')
  [✓] Input: 'the' → Predicted: 'mat' (should be: 'mat')
  [✓] Input: 'mat' → Predicted: 'because' (should be: 'because')
  [✓] Input: 'because' → Predicted: 'it' (should be: 'it')
  [✓] Input: 'it' → Pr

In [ ]:
# ─── Try it out ──────────────────────────────────────────────────────
# ─── Generation ──────────────────────────────────────────────────────

print("Greedy generation with KV cache:")
output = generate_with_cache(
    model, "the cat",
    word_to_id, id_to_word,
    max_new_tokens=8,
    do_sample=False,
)
print(f"\nFinal output: {output!r}")

Greedy generation with KV cache:
Starting prompt: 'the cat' → tokens: [6, 1]
  Prefill done. First generated token: 'sat'
  Step 2: 'on'
  Step 3: 'the'
  Step 4: 'mat'
  Step 5: 'because'
  Step 6: 'it'
  Step 7: 'was'
  Step 8: 'tired'

Final output: 'the cat sat on the mat because it was tired'


### Explanation

This loop has the same structure as the naive generation we wrote before, but with two crucial additions: a clearly separated **prefill** and **decode** phase, and the cache passed explicitly between calls.

---

**The prefill phase:**

```python
logits, kv_caches = model(
    token_ids,
    kv_caches=None,
    start_pos=0,
)
```

We feed the whole prompt at once with no cache (`kv_caches=None`). This is the only call where we process multiple tokens together. The model:

- Embeds all prompt tokens
- Runs the full forward pass through every block, building up K and V for each token at each block
- Returns logits for all positions AND the full initialized cache

We then pick the first generated token using `logits[-1]` — the last position of the prompt is what predicts what comes after the prompt.

This phase costs O(prompt_length²) work for attention — same as before. **This is where "time to first token" comes from.** If you submit a 1000-token prompt, the model has to process all 1000 tokens to build the cache before it can produce even one new token.

---

**The decode phase:**

```python
for step in range(max_new_tokens - 1):
    current_pos = len(generated_ids) - 1
    input_tensor = torch.tensor([next_token_id])

    logits, kv_caches = model(
        input_tensor,
        kv_caches=kv_caches,
        start_pos=current_pos,
    )
```

Each iteration feeds **one token** — the one we just generated. The cache from the previous call is passed in, and a new (one row larger) cache comes back.

**Why `max_new_tokens - 1`?** Because we already produced one token during prefill. If the user wants 5 new tokens total, we generate 4 more here. Off-by-one details that matter.

**Why `current_pos = len(generated_ids) - 1`?** When we've generated 3 tokens so far (`generated_ids` has length 3), the most recent one sits at position 2. That's `start_pos` for this iteration's positional encoding — the new token's location in the full sequence.

**Why `logits[-1]` even though logits is shape `(1, vocab_size)`?** When seq_len is 1, `logits[-1]` and `logits[0]` are the same thing — the only row. Using `[-1]` keeps the code consistent with prefill, where logits had multiple rows and we wanted the last one. **One pattern works for both phases.**

This is the "tokens per second" phase. Each iteration is cheap — O(seq_len_so_far) for attention, but O(1) for everything else (embedding, FFN, layer norms all run on just one token).

---